# 07 — Integração e Harmonização | Camada Prata

## Objetivo

Integrar as avaliações tratadas do IMDb e do Rotten Tomatoes
em uma estrutura analítica comum, preservando a origem e a
semântica dos dados de cada fonte.

## Fontes

- `mvp3_eng_dados.prata.imdb_avaliacoes`
- `mvp3_eng_dados.prata.rt_avaliacoes`
- `mvp3_eng_dados.prata.rt_filmes`

## Estratégia

IMDb:
- avaliação textual: `review`
- classificação original: `sentiment`
- sentimento harmonizado: `sentimento`

Rotten Tomatoes:
- avaliação textual: `review_content`
- classificação original: `review_type`
- sentimento harmonizado: `sentimento`

A harmonização permite analisar conjuntamente avaliações
positivas e negativas sem eliminar a classificação original
nem a identificação da fonte.

## Resultado

Será criada a tabela:

`mvp3_eng_dados.prata.avaliacoes_harmonizadas`

In [0]:
from pyspark.sql import functions as F

In [0]:
TABELA_IMDB = (
    "mvp3_eng_dados.prata.imdb_avaliacoes"
)

TABELA_RT_AVALIACOES = (
    "mvp3_eng_dados.prata.rt_avaliacoes"
)

TABELA_RT_FILMES = (
    "mvp3_eng_dados.prata.rt_filmes"
)

TABELA_HARMONIZADA = (
    "mvp3_eng_dados.prata.avaliacoes_harmonizadas"
)

print("IMDb:", TABELA_IMDB)
print("RT avaliações:", TABELA_RT_AVALIACOES)
print("RT filmes:", TABELA_RT_FILMES)
print("Destino:", TABELA_HARMONIZADA)

IMDb: mvp3_eng_dados.prata.imdb_avaliacoes
RT avaliações: mvp3_eng_dados.prata.rt_avaliacoes
RT filmes: mvp3_eng_dados.prata.rt_filmes
Destino: mvp3_eng_dados.prata.avaliacoes_harmonizadas


In [0]:
df_imdb = spark.table(
    TABELA_IMDB
)

df_rt_avaliacoes = spark.table(
    TABELA_RT_AVALIACOES
)

df_rt_filmes = spark.table(
    TABELA_RT_FILMES
)

print(
    f"IMDb: "
    f"{df_imdb.count():,}"
)

print(
    f"RT avaliações: "
    f"{df_rt_avaliacoes.count():,}"
)

print(
    f"RT filmes: "
    f"{df_rt_filmes.count():,}"
)

IMDb: 49,582
RT avaliações: 1,010,546
RT filmes: 17,712


In [0]:
print("=== IMDb ===")
df_imdb.printSchema()

print("\n=== Rotten Tomatoes — avaliações ===")
df_rt_avaliacoes.printSchema()

print("\n=== Rotten Tomatoes — filmes ===")
df_rt_filmes.printSchema()

=== IMDb ===
root
 |-- id_avaliacao: string (nullable = true)
 |-- review: string (nullable = true)
 |-- sentiment: string (nullable = true)
 |-- sentimento: string (nullable = true)
 |-- fonte_dados: string (nullable = true)
 |-- data_hora_processamento: timestamp (nullable = true)


=== Rotten Tomatoes — avaliações ===
root
 |-- id_avaliacao: string (nullable = true)
 |-- rotten_tomatoes_link: string (nullable = true)
 |-- critic_name: string (nullable = true)
 |-- top_critic: string (nullable = true)
 |-- eh_top_critic: boolean (nullable = true)
 |-- publisher_name: string (nullable = true)
 |-- review_type: string (nullable = true)
 |-- sentimento: string (nullable = true)
 |-- review_score: string (nullable = true)
 |-- review_date: string (nullable = true)
 |-- data_avaliacao: date (nullable = true)
 |-- review_content: string (nullable = true)
 |-- fonte_dados: string (nullable = true)
 |-- data_hora_processamento: timestamp (nullable = true)


=== Rotten Tomatoes — filmes ===
r

In [0]:
df_imdb_harmonizado = (
    df_imdb
    .select(
        F.col("id_avaliacao"),
        
        F.lit("IMDb")
        .alias("fonte"),
        
        F.lit("USUARIO")
        .alias("tipo_avaliador"),
        
        F.lit(None)
        .cast("string")
        .alias("id_filme"),
        
        F.lit(None)
        .cast("string")
        .alias("titulo_filme"),
        
        F.lit(None)
        .cast("string")
        .alias("link_filme"),
        
        F.col("review")
        .alias("texto_avaliacao"),
        
        F.col("sentiment")
        .alias("classificacao_original"),
        
        F.col("sentimento"),
        
        F.lit(None)
        .cast("string")
        .alias("nota_original"),
        
        F.lit(None)
        .cast("date")
        .alias("data_avaliacao"),
        
        F.lit(None)
        .cast("string")
        .alias("nome_avaliador"),
        
        F.lit(None)
        .cast("string")
        .alias("publicacao"),
        
        F.lit(None)
        .cast("boolean")
        .alias("eh_top_critic")
    )
)

In [0]:
df_rt_filmes_integracao = (
    df_rt_filmes
    .select(
        "rotten_tomatoes_link",
        "id_filme",
        "movie_title"
    )
    .dropDuplicates(
        ["rotten_tomatoes_link"]
    )
)

In [0]:
df_rt_enriquecido = (
    df_rt_avaliacoes
    .join(
        df_rt_filmes_integracao,
        on="rotten_tomatoes_link",
        how="left"
    )
)

qtd_rt_original = (
    df_rt_avaliacoes.count()
)

qtd_rt_enriquecido = (
    df_rt_enriquecido.count()
)

print(
    "RT antes do enriquecimento: "
    f"{qtd_rt_original:,}"
)

print(
    "RT após enriquecimento: "
    f"{qtd_rt_enriquecido:,}"
)

assert (
    qtd_rt_original
    == qtd_rt_enriquecido
), (
    "ERRO: o JOIN alterou a quantidade "
    "de avaliações do Rotten Tomatoes."
)

print(
    "\n✓ Cardinalidade do JOIN preservada."
)

RT antes do enriquecimento: 1,010,546
RT após enriquecimento: 1,010,546

✓ Cardinalidade do JOIN preservada.


In [0]:
df_rt_harmonizado = (
    df_rt_enriquecido
    .select(
        F.col("id_avaliacao"),
        
        F.lit("Rotten Tomatoes")
        .alias("fonte"),
        
        F.lit("CRITICO")
        .alias("tipo_avaliador"),
        
        F.col("id_filme"),
        
        F.col("movie_title")
        .alias("titulo_filme"),
        
        F.col("rotten_tomatoes_link")
        .alias("link_filme"),
        
        F.col("review_content")
        .alias("texto_avaliacao"),
        
        F.col("review_type")
        .alias("classificacao_original"),
        
        F.col("sentimento"),
        
        F.col("review_score")
        .alias("nota_original"),
        
        F.col("data_avaliacao"),
        
        F.col("critic_name")
        .alias("nome_avaliador"),
        
        F.col("publisher_name")
        .alias("publicacao"),
        
        F.col("eh_top_critic")
    )
)

In [0]:
colunas_imdb = (
    df_imdb_harmonizado.columns
)

colunas_rt = (
    df_rt_harmonizado.columns
)

print("Colunas IMDb:")
print(colunas_imdb)

print("\nColunas Rotten Tomatoes:")
print(colunas_rt)

assert (
    colunas_imdb
    == colunas_rt
), (
    "ERRO: os esquemas harmonizados "
    "não possuem as mesmas colunas."
)

print(
    "\n✓ Estruturas compatíveis."
)

Colunas IMDb:
['id_avaliacao', 'fonte', 'tipo_avaliador', 'id_filme', 'titulo_filme', 'link_filme', 'texto_avaliacao', 'classificacao_original', 'sentimento', 'nota_original', 'data_avaliacao', 'nome_avaliador', 'publicacao', 'eh_top_critic']

Colunas Rotten Tomatoes:
['id_avaliacao', 'fonte', 'tipo_avaliador', 'id_filme', 'titulo_filme', 'link_filme', 'texto_avaliacao', 'classificacao_original', 'sentimento', 'nota_original', 'data_avaliacao', 'nome_avaliador', 'publicacao', 'eh_top_critic']

✓ Estruturas compatíveis.


In [0]:
df_avaliacoes_harmonizadas = (
    df_imdb_harmonizado
    .unionByName(
        df_rt_harmonizado
    )
)

In [0]:
df_avaliacoes_harmonizadas = (
    df_avaliacoes_harmonizadas
    .withColumn(
        "id_avaliacao_global",
        F.sha2(
            F.concat_ws(
                "||",
                F.col("fonte"),
                F.col("id_avaliacao")
            ),
            256
        )
    )
)

In [0]:
df_avaliacoes_harmonizadas = (
    df_avaliacoes_harmonizadas
    .withColumn(
        "data_hora_integracao",
        F.current_timestamp()
    )
    .select(
        "id_avaliacao_global",
        "id_avaliacao",
        "fonte",
        "tipo_avaliador",
        "id_filme",
        "titulo_filme",
        "link_filme",
        "texto_avaliacao",
        "classificacao_original",
        "sentimento",
        "nota_original",
        "data_avaliacao",
        "nome_avaliador",
        "publicacao",
        "eh_top_critic",
        "data_hora_integracao"
    )
)

df_avaliacoes_harmonizadas.printSchema()

root
 |-- id_avaliacao_global: string (nullable = true)
 |-- id_avaliacao: string (nullable = true)
 |-- fonte: string (nullable = false)
 |-- tipo_avaliador: string (nullable = false)
 |-- id_filme: string (nullable = true)
 |-- titulo_filme: string (nullable = true)
 |-- link_filme: string (nullable = true)
 |-- texto_avaliacao: string (nullable = true)
 |-- classificacao_original: string (nullable = true)
 |-- sentimento: string (nullable = true)
 |-- nota_original: string (nullable = true)
 |-- data_avaliacao: date (nullable = true)
 |-- nome_avaliador: string (nullable = true)
 |-- publicacao: string (nullable = true)
 |-- eh_top_critic: boolean (nullable = true)
 |-- data_hora_integracao: timestamp (nullable = false)



In [0]:
qtd_imdb = (
    df_imdb_harmonizado.count()
)

qtd_rt = (
    df_rt_harmonizado.count()
)

qtd_esperada = (
    qtd_imdb + qtd_rt
)

qtd_integrada = (
    df_avaliacoes_harmonizadas.count()
)

print(
    f"IMDb: "
    f"{qtd_imdb:,}"
)

print(
    f"Rotten Tomatoes: "
    f"{qtd_rt:,}"
)

print(
    "Total esperado: "
    f"{qtd_esperada:,}"
)

print(
    "Total integrado: "
    f"{qtd_integrada:,}"
)

assert (
    qtd_integrada
    == qtd_esperada
), (
    "ERRO: quantidade integrada "
    "difere da soma das fontes."
)

print(
    "\n✓ Cardinalidade da integração validada."
)

IMDb: 49,582
Rotten Tomatoes: 1,010,546
Total esperado: 1,060,128
Total integrado: 1,060,128

✓ Cardinalidade da integração validada.


In [0]:
ids_globais_nulos = (
    df_avaliacoes_harmonizadas
    .filter(
        F.col(
            "id_avaliacao_global"
        ).isNull()
    )
    .count()
)

ids_globais_distintos = (
    df_avaliacoes_harmonizadas
    .select(
        "id_avaliacao_global"
    )
    .distinct()
    .count()
)

print(
    "IDs globais nulos: "
    f"{ids_globais_nulos:,}"
)

print(
    "IDs globais distintos: "
    f"{ids_globais_distintos:,}"
)

assert (
    ids_globais_nulos == 0
), (
    "ERRO: existem IDs globais nulos."
)

assert (
    ids_globais_distintos
    == qtd_integrada
), (
    "ERRO: id_avaliacao_global "
    "não é único."
)

print(
    "\n✓ Chave global validada."
)

IDs globais nulos: 0
IDs globais distintos: 1,060,128

✓ Chave global validada.


In [0]:
sentimentos_invalidos = (
    df_avaliacoes_harmonizadas
    .filter(
        F.col("sentimento").isNull()
        |
        ~F.col("sentimento").isin(
            "POSITIVO",
            "NEGATIVO"
        )
    )
    .count()
)

print(
    "Sentimentos inválidos: "
    f"{sentimentos_invalidos:,}"
)

assert (
    sentimentos_invalidos == 0
), (
    "ERRO: existem sentimentos "
    "fora do domínio harmonizado."
)

print(
    "\n✓ Domínio de sentimento validado."
)

Sentimentos inválidos: 0

✓ Domínio de sentimento validado.


In [0]:
display(
    df_avaliacoes_harmonizadas
    .groupBy(
        "fonte",
        "sentimento"
    )
    .count()
    .orderBy(
        "fonte",
        "sentimento"
    )
)

fonte,sentimento,count
IMDb,NEGATIVO,24698
IMDb,POSITIVO,24884
Rotten Tomatoes,NEGATIVO,366614
Rotten Tomatoes,POSITIVO,643932


In [0]:
inconsistencias_imdb = (
    df_avaliacoes_harmonizadas
    .filter(
        F.col("fonte") == "IMDb"
    )
    .filter(
        (
            F.lower(
                F.col(
                    "classificacao_original"
                )
            ) == "positive"
        )
        &
        (
            F.col("sentimento")
            != "POSITIVO"
        )
        |
        (
            F.lower(
                F.col(
                    "classificacao_original"
                )
            ) == "negative"
        )
        &
        (
            F.col("sentimento")
            != "NEGATIVO"
        )
    )
    .count()
)

inconsistencias_rt = (
    df_avaliacoes_harmonizadas
    .filter(
        F.col("fonte")
        == "Rotten Tomatoes"
    )
    .filter(
        (
            F.lower(
                F.col(
                    "classificacao_original"
                )
            ) == "fresh"
        )
        &
        (
            F.col("sentimento")
            != "POSITIVO"
        )
        |
        (
            F.lower(
                F.col(
                    "classificacao_original"
                )
            ) == "rotten"
        )
        &
        (
            F.col("sentimento")
            != "NEGATIVO"
        )
    )
    .count()
)

print(
    "Inconsistências IMDb: "
    f"{inconsistencias_imdb:,}"
)

print(
    "Inconsistências RT: "
    f"{inconsistencias_rt:,}"
)

assert inconsistencias_imdb == 0
assert inconsistencias_rt == 0

print(
    "\n✓ Harmonização semântica validada."
)

Inconsistências IMDb: 0
Inconsistências RT: 0

✓ Harmonização semântica validada.


In [0]:
display(
    df_avaliacoes_harmonizadas
    .groupBy("fonte")
    .agg(
        F.count("*")
        .alias("registros"),
        
        F.sum(
            F.col("id_filme")
            .isNull()
            .cast("int")
        )
        .alias("id_filme_nulo"),
        
        F.sum(
            F.col("data_avaliacao")
            .isNull()
            .cast("int")
        )
        .alias("data_avaliacao_nula"),
        
        F.sum(
            F.col("nome_avaliador")
            .isNull()
            .cast("int")
        )
        .alias("nome_avaliador_nulo"),
        
        F.sum(
            F.col("nota_original")
            .isNull()
            .cast("int")
        )
        .alias("nota_original_nula")
    )
)

fonte,registros,id_filme_nulo,data_avaliacao_nula,nome_avaliador_nulo,nota_original_nula
IMDb,49582,49582,49582,49582,49582
Rotten Tomatoes,1010546,130,0,16454,273605


In [0]:
(
    df_avaliacoes_harmonizadas
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(
        TABELA_HARMONIZADA
    )
)

print(
    "Tabela gravada:"
)

print(
    TABELA_HARMONIZADA
)

Tabela gravada:
mvp3_eng_dados.prata.avaliacoes_harmonizadas


In [0]:
df_validacao = (
    spark.table(
        TABELA_HARMONIZADA
    )
)

qtd_persistida = (
    df_validacao.count()
)

ids_persistidos = (
    df_validacao
    .select(
        "id_avaliacao_global"
    )
    .distinct()
    .count()
)

assert (
    qtd_persistida
    == qtd_integrada
), (
    "ERRO: quantidade persistida "
    "difere da integração."
)

assert (
    ids_persistidos
    == qtd_persistida
), (
    "ERRO: chave global não é "
    "única na tabela persistida."
)

print(
    "Registros persistidos: "
    f"{qtd_persistida:,}"
)

print(
    "IDs globais distintos: "
    f"{ids_persistidos:,}"
)

print(
    "\n✓ Persistência validada."
)

Registros persistidos: 1,060,128
IDs globais distintos: 1,060,128

✓ Persistência validada.


In [0]:
display(
    df_validacao
    .select(
        "id_avaliacao_global",
        "fonte",
        "tipo_avaliador",
        "titulo_filme",
        "texto_avaliacao",
        "classificacao_original",
        "sentimento",
        "nota_original",
        "data_avaliacao"
    )
    .orderBy(
        F.rand()
    )
    .limit(20)
)

id_avaliacao_global,fonte,tipo_avaliador,titulo_filme,texto_avaliacao,classificacao_original,sentimento,nota_original,data_avaliacao
5c4d4a194df2f921c87eb4aa1f26cc98a19cf1b68ab5b3f76922cf6b91347416,Rotten Tomatoes,CRITICO,Maleficent,You feel a little absurd being moved by something as shoddily put together as this. But it likes these women -- even the nincompoop pixies. It believes in its star.,Fresh,POSITIVO,null,2014-05-30
a234375a98940acc746001d1c25b2f03348d3e203552664edd44077c90294aee,Rotten Tomatoes,CRITICO,True Grit,This is what you find when you look up 'Western' in the cinematic genre dictionary.,Fresh,POSITIVO,5/5,2003-05-18
6d0350f7fa73b85a9acd5b947a46c0f556a2a5b8c2c2f93e0f017bd03110e1d8,Rotten Tomatoes,CRITICO,Tenderness,"Since moving to Hollywood, Australian director John Polson has become a very competent filmmaker. I hope he finds better scripts for his future projects.",Rotten,NEGATIVO,null,2009-05-01
9399f66c361c97e6b1855d4f8b4ebfe161cb4107960de8f20c9b2d21a32df18f,Rotten Tomatoes,CRITICO,All the Queen's Men,"Bad in such a bizarre way that it's almost worth seeing, if only to witness the crazy confluence of purpose and taste.",Rotten,NEGATIVO,null,2002-10-24
fc03c94cce903c9faf2d1a9d63c47ea752d46fa821c59b8d9279e40cc2a6df42,Rotten Tomatoes,CRITICO,Defending Your Life,One of Brooks' warmest and wittiest projects. Nice to see such a soft side of Streep.,Fresh,POSITIVO,4.5/5,2002-08-12
096355d2282b922c5f625191fe8315a484815c51833a42094e3e32a2a8e700d4,Rotten Tomatoes,CRITICO,Watchmen,"I didn't hate it. In fact, I'm pretty sure I liked it. I was just disappointed. I'm absolutely seeing it again, but disappointed nonetheless.",Fresh,POSITIVO,6/10,2009-02-25
04564a0dae6dd14f633d53f7085a2772bf73bc735a99e7b34230231c23a46a6a,Rotten Tomatoes,CRITICO,Oz the Great and Powerful,Looking backwards in more ways than one.,Rotten,NEGATIVO,null,2013-04-11
b0b7d5e05fd4b7fe47d1137ad43d14a9018efbba272e9220e33c2ea3cb671305,Rotten Tomatoes,CRITICO,Black Snake Moan,"Black Snake Moan finally, is a comedy of faith and faithlessness, in their religious and also their sexual dimensions...A consummate act of jerking our collective chain, and making us love it.",Fresh,POSITIVO,B+,2007-03-03
96d427b79c50ad444d876c845738ee6a1d3fc786949884d18fa54706f9e4f224,Rotten Tomatoes,CRITICO,Warcraft,"The performances are solid, given the banality of the material, and the special effects are impressive, but there is little in Warcraft that hasn't been seen before.",Rotten,NEGATIVO,null,2016-06-21
071c575b7938445def66d88e7d280f169165f8205cdab950902e42dfefe1978c,Rotten Tomatoes,CRITICO,Crime After Crime,"This emotionally affecting documentary will move you like few films this year, thanks to a luminous heroine and an unlikely but appealing trio who work tirelessly to release her from prison.",Fresh,POSITIVO,4/4,2011-08-04


In [0]:
print("=" * 60)

print(
    "INTEGRAÇÃO E HARMONIZAÇÃO "
    "DA CAMADA PRATA CONCLUÍDA"
)

print("=" * 60)

print(
    f"IMDb: "
    f"{qtd_imdb:,}"
)

print(
    f"Rotten Tomatoes: "
    f"{qtd_rt:,}"
)

print(
    f"Total integrado: "
    f"{qtd_integrada:,}"
)

print(
    "IDs globais distintos: "
    f"{ids_globais_distintos:,}"
)

print(
    "\nHarmonização:"
)

print(
    "✓ fontes preservadas"
)

print(
    "✓ classificações originais preservadas"
)

print(
    "✓ sentimentos harmonizados"
)

print(
    "✓ tipos de avaliador identificados"
)

print(
    "✓ dados RT enriquecidos com filmes"
)

print(
    "✓ chave global criada"
)

print(
    "✓ cardinalidade validada"
)

print(
    "✓ tabela Delta persistida"
)

INTEGRAÇÃO E HARMONIZAÇÃO DA CAMADA PRATA CONCLUÍDA
IMDb: 49,582
Rotten Tomatoes: 1,010,546
Total integrado: 1,060,128
IDs globais distintos: 1,060,128

Harmonização:
✓ fontes preservadas
✓ classificações originais preservadas
✓ sentimentos harmonizados
✓ tipos de avaliador identificados
✓ dados RT enriquecidos com filmes
✓ chave global criada
✓ cardinalidade validada
✓ tabela Delta persistida
